# 00 · Cuaderno maestro / *Master notebook*
**GUAIRA-SAR** · SRL *2026 Venezuela, Colombia and Panama Earthquakes*

🇪🇸 **Un solo cuaderno** que llama a todo el flujo (inventario → descargas → InSAR → daño → figuras) usando el código de GitHub (`src/`). Cada paso se ejecuta **una vez**: queda registrado en Drive (`05_documentacion/pasos_ejecutados.csv`) y se salta solo la próxima vez. Los pasos ya hechos el 19-nov figuran como ✅ *previo*. Para repetir uno a propósito: `P("id", forzar=True)`.
🇬🇧 **One notebook** that calls the whole workflow using the GitHub code. Each step runs **once**; it is logged on Drive and skipped next time. Steps done on 19 Nov show as ✅ *previo*. To repeat on purpose: `P("id", forzar=True)`.

⚠️ 🇪🇸 **HyP3:** el envío identifica trabajos por *par de escenas + resolución* (no por nombre) y **solo envía lo que no existe**. Hoy solo falta 033A a 40 m.
⚠️ 🇬🇧 **HyP3:** submissions are matched by *scene pair + resolution*; only what does not exist is sent. Today only 033A at 40 m is missing.
🔒 🇪🇸 Credenciales en **Secretos de Colab** (🔑 en la barra lateral): `EARTHDATA_USER` y `EARTHDATA_PASS`. No se escriben en el cuaderno. 🇬🇧 Credentials in **Colab Secrets**; never typed into the notebook.
📝 Contexto completo / full context: `docs/HANDOFF.md`.

## 0 · Preparar entorno / *Setup*

In [ ]:
NOTEBOOK = "00_Maestro"
!pip install -q asf_search hyp3_sdk rasterio geopandas shapely contextily pyarrow duckdb boto3 pystac-client requests h5py pyproj scipy huggingface_hub cmcrameri
import os, subprocess, datetime
REPO = "/content/SAR_multifrequency"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/jf-floresriera/SAR_multifrequency", REPO], check=True)
SRC = f"{REPO}/src"
VERSION = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR"
TRABAJO = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026"
for d in (TRABAJO, f"{RAIZ}/05_documentacion"):
    os.makedirs(d, exist_ok=True)
os.chdir(TRABAJO)
with open(f"{RAIZ}/05_documentacion/versiones_codigo.csv", "a") as f:   # trazabilidad / traceability
    f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M},{NOTEBOOK},{VERSION}\n")
import sys, importlib
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import comun, maestro; importlib.reload(comun); importlib.reload(maestro)
LOGS = f"{RAIZ}/05_documentacion/logs"
def correr(script):
    """Ejecuta src/<script> y guarda todo lo impreso en 05_documentacion/logs / runs and logs everything."""
    return comun.correr(script, LOGS, SRC, NOTEBOOK, VERSION)
def P(pid, **kw):
    """Ejecuta un paso del flujo salvo que ya esté hecho / run a workflow step unless already done."""
    return maestro.paso(pid, correr, RAIZ, VERSION, **kw)
print(f"Código / code: {VERSION} · Carpeta / folder: {os.getcwd()}")

## 1 · Credenciales NASA Earthdata (Secretos de Colab) / *Credentials (Colab Secrets)*

In [ ]:
import asf_search as asf
try:
    from google.colab import userdata
    usuario, clave = userdata.get("EARTHDATA_USER"), userdata.get("EARTHDATA_PASS")
except Exception:
    import getpass
    print("Sin Secretos de Colab: se pedirán aquí (la clave no se muestra). / No Colab Secrets: prompting here.")
    usuario = input("Usuario Earthdata / username: "); clave = getpass.getpass("Contraseña / password: ")
session = asf.ASFSession().auth_with_creds(usuario, clave)
print("Autenticación correcta / authenticated")

## 2 · Panel de estado / *Status board*
✅ hecho · ♻️ se puede correr siempre · ⏳ pendiente

In [ ]:
import pandas as pd; pd.set_option("display.max_colwidth", 90, "display.width", 200)
maestro.estado(RAIZ)

## A · Inventario multisensor / *Multi-sensor inventory*
🇪🇸 Umbra (X), NISAR y ALOS-2 (L), Sentinel-1 (C), Planet. Estas figuras y tablas van al **material suplementario** (S1–S3); una versión sintética alimenta la Fig. 1. 🇬🇧 These go to the **electronic supplement**.

In [ ]:
P("inv_umbra")
P("inv_nisar")

In [ ]:
P("inv_s1", variables={"ENVIAR_HYP3": False})   # solo busca / search only

In [ ]:
P("inv_alos2")   # ALOS-2: si ASF no lo tiene, ver docs/HANDOFF.md §5 (fecha de corte 5-dic) / see HANDOFF §5

In [ ]:
P("inv_tabla")
P("mapas_base", forzar=True)   # forzar=True: repite aunque ya esté hecho (código nuevo) / re-run even if done (new code)

## B · HyP3 y descargas / *HyP3 and downloads*
🇪🇸 `hyp3_estado` descarga lo terminado a Drive (HyP3 borra a los ~14 días). `hyp3_envio` está **bloqueado** hasta poner `confirmar=True`: enviará SOLO lo que no exista (hoy: 033A a 40 m, 1 trabajo). 🇬🇧 `hyp3_envio` is locked until `confirmar=True`.

In [ ]:
P("hyp3_estado", variables={"DESCARGAR": True})

In [ ]:
P("hyp3_envio", confirmar=False, variables={"ENVIAR_HYP3": True})   # ← cambiar a confirmar=True para enviar / set True to submit

In [ ]:
P("descargas", variables={"ESPERAR_HYP3": False})   # NISAR (ya en Drive se omite), Umbra GEC, Sentinel-1

## C · InSAR banda C y L / *C- and L-band InSAR*
🇪🇸 Escriba las coordenadas de CCS1 (ficha NGL / IGVSB) para validar con GNSS. 🇬🇧 Enter CCS1 coordinates to validate.

In [ ]:
P("auxiliares")
P("nisar_tif")
P("s1_tif")

In [ ]:
P("descomp", variables={"CCS1_LON": None, "CCS1_LAT": None})   # ej./e.g. -66.88, 10.49 (verificar)

## D · Daño: Umbra X + Planet + referencia / *Damage*
🇪🇸 Las figuras por sitio del 19-nov **no son publicables** (jet, título interno, Esri): tras actualizar `umbra_planet_sitios.py` al estilo nuevo use `forzar=True`. 🇬🇧 The 19 Nov per-site figures are not publishable; re-run with `forzar=True` after the style update.

In [ ]:
P("danio_ref")

In [ ]:
P("danio_sitios", variables={"LADO_M": 1000, "ZOOM_M": 400, "SITIOS_A_PROCESAR": ["Catia_La_Mar", "Playa_Grande_Urimare", "La_Guaira"]})

## E · Figuras y tablas finales / *Final figures and tables*
🇪🇸 **Texto principal (≤ 10 figuras, ≤ 3 tablas):** Fig. 1 ubicación y cobertura · Fig. 3–5 LOS y Este/Vertical C vs L · Fig. 6 coherencia por cobertura · Fig. 7 ventana postsísmica · Fig. 8 daño Umbra × Planet · Fig. 9 daño vs gradiente/distancia a la falla · Fig. 10 infraestructura · Tablas 1–3.
**Suplemento (sin límite):** S1 línea de tiempo · S2 huellas por sensor (Sentinel-1, NISAR, ALOS-2, Umbra, Planet) · Tabla S1 inventario completo · S3 los 6 sitios de daño · S4 paletas y pruebas de gris.
🇬🇧 Scripts for Figs. 6, 7, 9, 10 and Table 3 are **pending**; they will be added to `src/` and as steps of this notebook (`PASOS` in `src/maestro.py`).

In [ ]:
P("exportar")   # figuras a GitHub: NO subir imágenes Planet (CC BY-NC) ni Esri / do not upload Planet or Esri imagery

## Estado final / *Final status*

In [ ]:
maestro.estado(RAIZ)